In [ ]:
# =====================
# Configuration
# =====================
# (단일 파일 테스트용)
IO_PATH = r".\raw_data\p01\p01_0_0_arousal_2_4_0.csv"
IO_FMT = "csv"
IO_SEP = ","
IO_HEADER = 0
IO_SKIPROWS = [1]

# Shimmer 원본 컬럼명
TIMESTAMP_COL = "Shimmer_9F46_TimestampSync_Unix_CAL"
PPG_COL       = "Shimmer_9F46_PPG_A13_CAL"
EDA_COL       = "Shimmer_9F46_GSR_Skin_Conductance_CAL"

# 파일 내부 라벨(없으면 None)
AROUSAL_COL = None
CONTENT_COL = None

# Sampling
FS_SPEC = 128.0


# Windowing
WIN_SEC = 5.0
OVERLAP = 0.5

# Filters / NeuroKit
PPG_BANDPASS = (0.5, 4.0, 4)  # (low, high, order)
EDA_HIGHPASS = (0.05, 4)
EDA_LOWPASS = (0.05, 4)

NK_CLEAN_METHOD  = "neurokit"
NK_PHASIC_METHOD = "cvxeda"

# Batch I/O
RAW_DIR = r"./raw_data/p06"
OUT_DIR = r"./dataset/ml_dataset"
OUT_CSV = "features_p06_arousal.csv"


In [9]:
# =====================
# Imports
# =====================
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Tuple
from scipy.signal import butter, filtfilt, find_peaks
from scipy.stats import kurtosis
import neurokit2 as nk

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 200)

# =====================
# Utilities
# =====================
def _read_any(path, fmt, sep, header, skiprows):
    if fmt == "csv":
        return pd.read_csv(path, sep=sep, header=header, skiprows=skiprows)
    elif fmt == "tsv":
        return pd.read_csv(path, sep="\t", header=header, skiprows=skiprows)
    elif fmt == "xlsx":
        return pd.read_excel(path, sheet_name=0, header=header)
    else:
        raise ValueError("fmt must be csv|tsv|xlsx")

def bandpass_ppg(x, fs, f_lo=0.5, f_hi=4.0, order=4):
    x = np.asarray(x, dtype=float)
    x = np.nan_to_num(x, nan=np.nanmedian(x))
    nyq = 0.5 * fs
    lo = np.clip(f_lo / nyq, 1e-6, 0.999)
    hi = np.clip(f_hi / nyq, 1e-6, 0.999)
    hi = max(hi, lo + 1e-6)
    b, a = butter(order, [lo, hi], btype='bandpass')
    return filtfilt(b, a, x)

def make_time_windows(t_sec: np.ndarray, win: float, overlap: float):
    """시간(초) 기반 윈도 인덱스 경계 생성(상대시간)."""
    if t_sec is None or len(t_sec) == 0:
        return []
    t0 = float(t_sec[0])
    t_rel = t_sec - t0
    hop = win * (1.0 - overlap)
    if hop <= 0:
        raise ValueError("overlap too large")
    out, s, t_end = [], 0.0, float(t_rel[-1])
    while s + win <= t_end + 1e-9:
        out.append((s, s + win))
        s += hop
    return out

def read_two_header_as_standard_df(path, sep=None):
    """Shimmer 2줄 헤더(csv/tsv) → 표준컬럼(timestamp, ppg, gsr)로 변환."""
    NAME_TS  = "Shimmer_9F46_TimestampSync_Unix_CAL"
    NAME_PPG = "Shimmer_9F46_PPG_A13_CAL"
    NAME_EDA = "Shimmer_9F46_GSR_Skin_Conductance_CAL"
    UNIT_TS, UNIT_PPG, UNIT_EDA = "ms", "mV", "uS"

    if sep is None:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            first = f.readline()
        sep = "\t" if "\t" in first else ","

    df0 = pd.read_csv(path, sep=sep, header=[0, 1], engine="python")
    df0.columns = pd.MultiIndex.from_tuples([(str(a).strip(), str(b).strip()) for a, b in df0.columns])
    df = pd.DataFrame({
        "timestamp": pd.to_numeric(df0[(NAME_TS,  UNIT_TS)],  errors="coerce"),
        "ppg":       pd.to_numeric(df0[(NAME_PPG, UNIT_PPG)], errors="coerce"),
        "gsr":       pd.to_numeric(df0[(NAME_EDA, UNIT_EDA)], errors="coerce"),
    })
    return df.dropna(subset=["timestamp", "ppg", "gsr"]).reset_index(drop=True)

def _parse_filename_meta(path_str: str):
    """
    파일명 규약 예: p01_34_0_arousal_0_21_0.csv
    parts: 0=p01, 1=trial, 2=file_state, 3='arousal', 4=pre, 5=content_id, 6=post
    """
    name = Path(path_str).stem
    parts = name.split("_")
    if len(parts) < 7:
        return {}
    return {
        "subject":      parts[0],
        "trial":        int(parts[1]),
        "file_state":   int(parts[2]),
        "pre_arousal":  int(parts[4]),
        "content_id":   int(parts[5]),
        "post_arousal": int(parts[6]),
    }

def trim_windows_by_state(windows_idx, file_state: int, target_n: int = 25):
    """file_state: 0=그대로, 1=앞 keep, 2=뒤 keep, 3=중앙 keep (개수=target_n)."""
    n = len(windows_idx)
    if n == 0 or file_state == 0:
        return windows_idx
    keep = min(target_n, n)
    if file_state == 1:
        return windows_idx[:keep]
    elif file_state == 2:
        return windows_idx[-keep:]
    elif file_state == 3:
        if n <= keep:
            return windows_idx
        start = (n - keep) // 2
        end = start + keep
        return windows_idx[start:end]
    return windows_idx

def _normalize_ts_to_ms(t_raw: np.ndarray) -> np.ndarray:
    """타임스탬프가 초 단위이면 ms로 변환해 통일."""
    t_raw = np.asarray(t_raw, dtype=float)
    med = float(np.nanmedian(t_raw)) if t_raw.size else 0.0
    # 흔한 기준: 초 단위면 중앙값이 1e7 미만인 경우가 많음
    return (t_raw * 1000.0) if med < 1e7 else t_raw




In [10]:
# =====================
# Feature Extraction
# =====================
def extract_ppg_features_from_segments(X: np.ndarray, fs: float = 128.0) -> pd.DataFrame:
    """PPG 시간영역 15 피처."""
    cols = [
        'ppg_max',
        'ppg_cumulative_max_mean','ppg_cumulative_min_mean',
        'ppg_min_to_max',
        'ppg_peak_to_rms','ppg_kurtosis',
        'ppg_mean_rise_time',
        'ppg_pulse_height_min','ppg_pulse_height_max',
        'ppg_heart_rate',
        'ppg_nn_mean','ppg_nn_std','ppg_rmssd','ppg_pnn50','ppg_pnn20'
    ]
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X[None, :]
    if X.size == 0:
        return pd.DataFrame(columns=cols)

    feats = []
    for x in X:
        x = np.nan_to_num(np.asarray(x, float), nan=np.nanmedian(x))

        # 기본 통계
        x_max = float(np.max(x)) if x.size else np.nan
        x_min = float(np.min(x)) if x.size else np.nan
        min_to_max = x_max - x_min if np.isfinite(x_max) and np.isfinite(x_min) else np.nan
        rms_v = float(np.sqrt(np.mean(x**2))) if x.size else np.nan
        peak_to_rms = float(x_max / rms_v) if (np.isfinite(x_max) and np.isfinite(rms_v) and rms_v > 0) else np.nan
        m = float(np.mean(x)) if x.size else np.nan
        v = float(np.mean((x - m)**2)) if x.size else np.nan
        kurt = float(np.mean((x - m)**4) / (v**2) - 3.0) if (x.size and v is not None and v > 0) else np.nan
        cmax_mean = float(np.mean(np.maximum.accumulate(x))) if x.size else np.nan
        cmin_mean = float(np.mean(np.minimum.accumulate(x))) if x.size else np.nan

        # 피크 추정
        distance = max(1, int(round(0.30 * fs)))
        prom = float(np.quantile(np.abs(x), 0.60)) if x.size else 0.0
        prom = max(prom, 1e-6)
        peaks, _ = find_peaks(x, distance=distance, prominence=prom)
        troughs, _ = find_peaks(-x, distance=distance, prominence=prom)

        # HR/HRV
        if len(peaks) >= 2:
            rr = np.diff(peaks) / float(fs)
            nn_mean = float(np.mean(rr)) if rr.size else np.nan
            nn_std  = float(np.std(rr, ddof=0)) if rr.size else np.nan
            if rr.size >= 2:
                drr = np.diff(rr)
                rmssd = float(np.sqrt(np.mean(drr**2)))
                pnn50 = float(np.mean(np.abs(drr) > 0.050))
                pnn20 = float(np.mean(np.abs(drr) > 0.020))
            else:
                rmssd = pnn50 = pnn20 = np.nan
            heart_rate = float(60.0 / nn_mean) if (nn_mean is not None and nn_mean > 0) else np.nan
        else:
            nn_mean = nn_std = rmssd = pnn50 = pnn20 = heart_rate = np.nan

        # 펄스 높이/상승시간
        pulse_heights, rise_times = [], []
        if len(peaks) and len(troughs):
            ti = 0
            for p in peaks:
                while ti + 1 < len(troughs) and troughs[ti + 1] < p:
                    ti += 1
                if ti < len(troughs) and troughs[ti] < p:
                    pulse_heights.append(x[p] - x[troughs[ti]])
                    rise_times.append((p - troughs[ti]) / float(fs))
        pulse_height_min = float(np.min(pulse_heights)) if pulse_heights else np.nan
        pulse_height_max = float(np.max(pulse_heights)) if pulse_heights else np.nan
        mean_rise_time   = float(np.mean(rise_times)) if rise_times else np.nan

        feats.append([
            x_max,
            cmax_mean, cmin_mean,
            min_to_max,
            peak_to_rms, kurt,
            mean_rise_time,
            pulse_height_min, pulse_height_max,
            heart_rate,
            nn_mean, nn_std, rmssd, pnn50, pnn20
        ])

    return pd.DataFrame(feats, columns=cols)

def scl_features_for_window(t_win, x_win):
    """SCL(tonic) 통계."""
    eps = 1e-12
    x = np.asarray(x_win, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {k: np.nan for k in [
            "scl_mean","scl_std","scl_median","scl_max",
            "scl_cumulative_max_end","scl_cumulative_min_end",
            "scl_min_to_max","scl_peak_to_rms","scl_kurtosis_excess"
        ]}
    mean_v = float(np.mean(x))
    std_v  = float(np.std(x, ddof=1)) if x.size > 1 else 0.0
    mdn_v  = float(np.median(x))
    max_v  = float(np.max(x))
    min_v  = float(np.min(x))
    range_v = max_v - min_v
    rms_v  = float(np.sqrt(np.mean(x * x)))
    peak_to_rms = float(max_v / (rms_v + eps)) if rms_v > 0 else np.nan
    kurt_ex = float(kurtosis(x, fisher=True, bias=False)) if x.size > 3 else np.nan
    cmax = np.maximum.accumulate(x)
    cmin = np.minimum.accumulate(x)
    return {
        "scl_mean": mean_v,
        "scl_std": std_v,
        "scl_median": mdn_v,
        "scl_max": max_v,
        "scl_cumulative_max_end": float(cmax[-1]),
        "scl_cumulative_min_end": float(cmin[-1]),
        "scl_min_to_max": range_v,
        "scl_peak_to_rms": peak_to_rms,
        "scl_kurtosis_excess": kurt_ex,
    }

def scr_features_for_window(t_win, x_win):
    """SCR(phasic) 통계 + 적분/전력."""
    eps = 1e-12
    x = np.asarray(x_win, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {k: np.nan for k in [
            "scr_std","scr_median","scr_max",
            "scr_cumulative_max_end","scr_cumulative_min_end",
            "scr_min_to_max","scr_peak_to_rms","scr_kurtosis_excess",
            "scr_integral_abs","scr_avg_power","scr_norm_avg_power",
            "scr_rms","scr_norm_rms"
        ]}
    std_v  = float(np.std(x, ddof=1)) if x.size > 1 else 0.0
    mdn_v  = float(np.median(x))
    max_abs = float(np.max(np.abs(x)))
    min_v   = float(np.min(x))
    max_v   = float(np.max(x))
    range_v = max_v - min_v
    cmax = np.maximum.accumulate(x)
    cmin = np.minimum.accumulate(x)
    rms_v = float(np.sqrt(np.mean(x * x)))
    peak_to_rms = float(max_abs / (rms_v + eps)) if (np.isfinite(max_abs) and np.isfinite(rms_v)) else np.nan
    kurt_ex = float(kurtosis(x, fisher=True, bias=False)) if x.size > 3 else np.nan

    # |x| 적분(시간축 정상인 경우 trapezoid)
    integral_abs = np.nan
    if t_win is not None:
        t = np.asarray(t_win, dtype=float)
        mask = np.isfinite(t)
        if mask.sum() >= 2:
            t = t[mask]
            n = min(len(t), len(x))
            t = t[:n]; x_int = x[:n]
            if not np.all(np.diff(t) > 0):
                order = np.argsort(t)
                t = t[order]; x_int = x_int[order]
            t_rel = t - t[0]
            if np.all(np.diff(t_rel) > 0):
                integral_abs = float(np.trapezoid(np.abs(x_int), t_rel))
    if not np.isfinite(integral_abs):
        if x.size >= 2:
            integral_abs = float(np.sum(np.abs((x[:-1] + x[1:]) / 2.0)))
        else:
            integral_abs = np.nan

    avg_power = float(np.mean(x * x))
    norm_avg_power = float(avg_power / (max_abs**2 + eps)) if np.isfinite(max_abs) else np.nan
    norm_rms = float(rms_v / (max_abs + eps)) if np.isfinite(max_abs) else np.nan

    return {
        "scr_std": std_v,
        "scr_median": mdn_v,
        "scr_max": max_v,
        "scr_cumulative_max_end": float(cmax[-1]),
        "scr_cumulative_min_end": float(cmin[-1]),
        "scr_min_to_max": range_v,
        "scr_peak_to_rms": peak_to_rms,
        "scr_kurtosis_excess": kurt_ex,
        "scr_integral_abs": integral_abs,
        "scr_avg_power": avg_power,
        "scr_norm_avg_power": norm_avg_power,
        "scr_rms": rms_v,
        "scr_norm_rms": norm_rms,
    }


In [11]:
# =====================
# Core per-file processing
# =====================
def process_one_file(path: str, sep=None) -> pd.DataFrame:
    """단일 파일 → 5s/50% 윈도별 PPG+EDA 피처 DF."""
    # 2줄 헤더 Shimmer 로더 사용(기존 함수 유지)
    df_raw = read_two_header_as_standard_df(path, sep=sep)  # columns: timestamp(ms|s), ppg, gsr

    # --- 타임스탬프 단위 정규화(+ 유효 FS 추정) ---
    t_ms = _normalize_ts_to_ms(df_raw["timestamp"].to_numpy(dtype=float))
    eff_fs = FS_SPEC  # 파일 별 실제 표본속도

    # --- 필터 & EDA 분해: eff_fs로 수행 ---
    f_lo, f_hi, p_order = PPG_BANDPASS
    ppg_f = bandpass_ppg(df_raw["ppg"].to_numpy(dtype=float), fs=eff_fs, f_lo=f_lo, f_hi=f_hi, order=p_order)

    x = np.asarray(df_raw["gsr"].to_numpy(dtype=float), dtype="float64").ravel()
    if np.any(~np.isfinite(x)):
        m = np.nanmean(x); x = np.nan_to_num(x, nan=(0.0 if not np.isfinite(m) else float(m)))
    x_clean = nk.eda_clean(x, sampling_rate=eff_fs, method=NK_CLEAN_METHOD)

    def _get_scl_scr(xc, fs, method):
        decomp, _ = nk.eda_phasic(xc, sampling_rate=fs, method=method)
        if isinstance(decomp, pd.DataFrame):
            return decomp["EDA_Tonic"].to_numpy(), decomp["EDA_Phasic"].to_numpy()
        if isinstance(decomp, dict):
            return np.asarray(decomp["EDA_Tonic"]), np.asarray(decomp["EDA_Phasic"])
        raise TypeError(f"Unexpected type: {type(decomp)}")

    try:
        scl, scr = _get_scl_scr(x_clean, eff_fs, NK_PHASIC_METHOD)
    except Exception:
        signals, _ = nk.eda_process(x, sampling_rate=eff_fs, method="neurokit")
        scl, scr = signals["EDA_Tonic"].to_numpy(), signals["EDA_Phasic"].to_numpy()

    work = pd.DataFrame({
        "timestamp": t_ms,  # ms
        "ppg": df_raw["ppg"].to_numpy(dtype=float),
        "ppg_f": ppg_f,
        "scl": scl,
        "scr": scr,
    }).dropna().reset_index(drop=True)
    if work.empty:
        return pd.DataFrame()

    # --- 윈도우 구축: 우선 '시간 기반', 실패 시 '샘플 기반(eff_fs)' 폴백 ---
    t_rel = (work["timestamp"].to_numpy(dtype=float) - float(work["timestamp"].iloc[0])) / 1000.0  # sec
    windows_time = make_time_windows(t_rel, WIN_SEC, OVERLAP)

    windows_idx = []
    for s_rel, e_rel in windows_time:
        s_idx = int(np.searchsorted(t_rel, s_rel, side="left"))
        e_idx = int(np.searchsorted(t_rel, e_rel, side="left"))
        if e_idx - s_idx >= 3:
            windows_idx.append((s_idx, e_idx))

    if not windows_idx:
        # 샘플 기반 폴백(실제 fs 기준)
        n = len(work)
        win = max(1, int(round(WIN_SEC * eff_fs)))
        hop = max(1, int(round(win * (1.0 - OVERLAP))))
        if n >= win:
            windows_idx = [(s, s + win) for s in range(0, n - win + 1, hop)]

    # 파일 상태 기반 트리밍
    meta = _parse_filename_meta(path)
    windows_idx = trim_windows_by_state(windows_idx, int(meta.get("file_state", 0)), target_n=25)

    # --- 최소 샘플 수: eff_fs 기준으로 동적 설정(윈도 길이의 25%만 있어도 허용) ---
    min_ppg = max(3, int(WIN_SEC * eff_fs * 0.25))

    rows = []
    for s_idx, e_idx in windows_idx:
        w = work.iloc[s_idx:e_idx]
        if len(w) < min_ppg:
            continue

        # PPG 피처 (fs=eff_fs)
        x_ppg = w["ppg_f"].to_numpy()
        if x_ppg.size < 3:
            continue
        ppg_feat = extract_ppg_features_from_segments(np.expand_dims(x_ppg, 0), fs=eff_fs).iloc[0].to_dict()

        # EDA 피처
        t_win_ms = w["timestamp"].to_numpy()
        scl_feat = scl_features_for_window(t_win_ms, w["scl"].to_numpy())
        scr_feat = scr_features_for_window(t_win_ms, w["scr"].to_numpy())

        rows.append({
            "timestamp": float(t_win_ms[0] / 1000.0),   # 창 시작(초)
            "win_start_s": float(s_idx / eff_fs),
            "win_end_s":   float(e_idx / eff_fs),
            **ppg_feat, **scl_feat, **scr_feat,
            **meta, "file_name": Path(path).name,
        })

    # 디버그(필요 시 주석 해제)
    # print(f"[DBG] {Path(path).name} | N={len(work)} | eff_fs={eff_fs:.1f}Hz | win={len(windows_idx)} | min_ppg={min_ppg}")

    return pd.DataFrame(rows) if rows else pd.DataFrame()


In [12]:
# =====================
# Single-file quick test (optional)
# =====================
def quick_test_single():
    df0 = _read_any(IO_PATH, IO_FMT, IO_SEP, IO_HEADER, IO_SKIPROWS)
    # 컬럼명 표준화
    rename_map = {TIMESTAMP_COL: "timestamp", PPG_COL: "ppg", EDA_COL: "gsr"}
    for k in [TIMESTAMP_COL, PPG_COL, EDA_COL]:
        if k not in df0.columns:
            raise KeyError(f"Required column missing: {k}")
    df = df0[[TIMESTAMP_COL, PPG_COL, EDA_COL]].rename(columns=rename_map)
    df = df.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
    df.to_csv(Path(OUT_DIR) / "quick_test_clean_preview.csv", index=False, encoding="utf-8-sig")
    print("[quick_test] saved preview:", (Path(OUT_DIR) / "quick_test_clean_preview.csv").resolve())


In [13]:
# # =====================
# # Single-subject Batch run
# # =====================
# def main():
#     out_dir = Path(OUT_DIR); out_dir.mkdir(parents=True, exist_ok=True)

#     p_dir = Path(RAW_DIR)
#     all_files = sorted([fp for fp in p_dir.glob("*.csv") if "_arousal_" in fp.name],
#                        key=lambda p: int(p.stem.split("_")[1]))
#     print(f"[INFO] found {len(all_files)} arousal files in {p_dir}")

#     all_rows = []
#     for i, fp in enumerate(all_files, 1):
#         print(f"[{i}/{len(all_files)}] {fp.name} ...", end="")
#         try:
#             df_feat = process_one_file(str(fp), sep=None)
#             print(f" rows: {len(df_feat)}")
#             if not df_feat.empty:
#                 all_rows.append(df_feat)
#         except Exception as e:
#             print(f" ERROR: {e}")

#     combined_all = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
#     out_path = Path(OUT_DIR) / OUT_CSV
#     combined_all.to_csv(out_path, index=False, encoding="utf-8-sig")
#     print(f"[SAVED] {out_path.resolve()}  (shape={combined_all.shape})")

# if __name__ == "__main__":
#     main()
#     # 필요 시 단일 파일 점검:
#     # quick_test_single()

In [14]:
# =====================
# Multi-subject Batch run (p01..pNN)
# =====================

RAW_ROOT = Path("./raw_data")     
OUT_ROOT = Path("./dataset/ml_dataset")  
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def _subject_code(idx: int) -> str:
    """1 -> 'p01', 12 -> 'p12'"""
    return f"p{idx:02d}"

def _sorted_arousal_files(p_dir: Path):
    files = [fp for fp in p_dir.glob("*.csv") if "_arousal_" in fp.name]
    def trial_index(fp: Path) -> int:
        try:
            return int(fp.stem.split("_")[1])
        except Exception:
            return 10**9  # 예외는 뒤로
    return sorted(files, key=trial_index)

def process_subject_folder(subj: str) -> pd.DataFrame:
    #싱글 폴더 to 피쳐 파일
    p_dir = RAW_ROOT / subj
    if not p_dir.exists():
        print(f"[WARN] {subj}: folder not found → skip")
        return pd.DataFrame()

    files = _sorted_arousal_files(p_dir)
    print(f"[INFO] {subj}: found {len(files)} arousal files in {p_dir}")

    rows = []
    for i, fp in enumerate(files, 1):
        print(f"  [{i}/{len(files)}] {fp.name} ...", end="")
        try:
            df_feat = process_one_file(str(fp), sep=None)
            print(f" rows: {len(df_feat)}")
            if not df_feat.empty:
                rows.append(df_feat)
        except Exception as e:
            print(f" ERROR: {e}")

    df_subj = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    out_path = OUT_ROOT / f"features_{subj}_arousal.csv"
    df_subj.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"[SAVED] {out_path.resolve()} (shape={df_subj.shape})")
    return df_subj

def run_multi_subjects(n_subjects: int, start_idx: int = 1) -> pd.DataFrame:
    #한 파일로 merge 
    all_df = []
    for sid in range(start_idx, start_idx + int(n_subjects)):
        subj = _subject_code(sid)
        df_subj = process_subject_folder(subj)
        if not df_subj.empty:
            all_df.append(df_subj)

    df_all = pd.concat(all_df, ignore_index=True) if all_df else pd.DataFrame()
    out_all = OUT_ROOT / f"features_{_subject_code(start_idx)}_to_{_subject_code(start_idx + int(n_subjects) - 1)}_arousal.csv"
    df_all.to_csv(out_all, index=False, encoding="utf-8-sig")
    print(f"[SAVED] {out_all.resolve()} (shape={df_all.shape})")
    return df_all

if __name__ == "__main__":
    try:
        n_in = int(input("How many folders?").strip())
    except Exception:
        n_in = 1
    run_multi_subjects(n_in, start_idx=1)


[INFO] p01: found 18 arousal files in raw_data\p01
  [1/18] p01_0_0_arousal_2_4_0.csv ... rows: 25
  [2/18] p01_3_0_arousal_1_28_1.csv ... rows: 25
  [3/18] p01_6_0_arousal_2_0_0.csv ... rows: 26
  [4/18] p01_9_0_arousal_1_13_0.csv ... rows: 25
  [5/18] p01_12_0_arousal_1_15_0.csv ... rows: 25
  [6/18] p01_15_0_arousal_0_50_0.csv ... rows: 25
  [7/18] p01_18_0_arousal_0_37_0.csv ... rows: 25
  [8/18] p01_21_0_arousal_0_35_0.csv ... rows: 25
  [9/18] p01_24_0_arousal_0_42_0.csv ... rows: 27
  [10/18] p01_28_0_arousal_0_20_0.csv ... rows: 24
  [11/18] p01_31_0_arousal_0_25_0.csv ... rows: 26
  [12/18] p01_34_0_arousal_0_21_0.csv ... rows: 24
  [13/18] p01_37_0_arousal_0_41_0.csv ... rows: 25
  [14/18] p01_40_0_arousal_1_8_0.csv ... rows: 25
  [15/18] p01_43_0_arousal_1_52_0.csv ... rows: 25
  [16/18] p01_46_0_arousal_0_30_0.csv ... rows: 25
  [17/18] p01_49_0_arousal_1_45_0.csv ... rows: 25
  [18/18] p01_52_0_arousal_2_11_0.csv ... rows: 25
[SAVED] C:\Users\jaebb\steam\dataset\ml_dataset